# Historical development screen: single-test-v1
**Not the recommended application workflow.** The real screen was stopped
after repeated rejection of complete multi-test outputs by the one-test rule.
It did not establish an efficacy result. The reviewed working application
subset is documented in `research/application/14b_genty_curated_20261117/README.md`.
This notebook is retained to reproduce the experimental configuration.

Enable Internet and T4 x2. This runs only two genty modules, both arms, seed42,
one round (four module trials). It never launches the larger pilot.

The notebook ships the source snapshot and audited real greedy initialization
evidence. It prepares new environments, pins the original real model revisions,
checks production/model/runtime identity before initialization-only reuse, and
requires **fresh GPU calibration**. Generated tests/repairs/outcomes are never
imported for scoring. Earlier cold costs remain separate. Old summaries can be
wrong or truncated; reusing them does not verify their semantics.

Both arms use the shared single-test-v1 output contract. This is an adapted
upstream baseline, not identical raw upstream prompting. A 60-minute shared
stress+small deadline includes weights/embedding downloads and model loads;
CPU bootstrap/preparation is separate. All failures and missing values remain.
This last cell waits for completion so Run All/Save Version can retain outputs.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORKDIR=Path('/kaggle/working')/('Test4Py_upstream_'+STAMP)
ROOT=Path('/kaggle/working')/('UPSTREAM_NF4_FRESH_COMPACT_'+STAMP)
subprocess.run(['git','clone','--depth','1','https://github.com/nguyendevwk/Test4Py-Setup-pilot-temp.git',str(WORKDIR)],check=True)
BASE_COMMIT='7a97eb0a6c8fefc2e8154d4c0b4487b4ae2b0ea3'
if subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()!=BASE_COMMIT:
    subprocess.run(['git','-C',str(WORKDIR),'fetch','--depth','1','origin',BASE_COMMIT],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'checkout','--detach','FETCH_HEAD'],check=True)
assert subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()==BASE_COMMIT
HASHES={'ablation_runner.py': '2e139f4b8f083cb7d83157326ee39ed451d93fb5765a1facca0c55c95864154a', 'analyze.py': '50f8d6c65b26922b2df1a453317bc1d6bc1e279614c810404d7a389ffd3946dd', 'benchmark19.py': 'ad74c1b19dde7d6a09f3417040a79032eda8af18d8288f56feccc69f5fa840d1', 'benchmark_worker.py': 'cc641933954af2bd15d0aecc45f04f44de7664ec0941b32b16d918b12493fb95', 'bootstrap_kaggle.py': 'd432da528c8bbcf92647bfe09c734ac8c638e051a43a7eac0307cce42e44735c', 'bootstrap_modern.py': '3c591e4399847fa5d556e507ef8177fd5dbb9e2e8fd4049b5b3afdac9628fed5', 'check_environment.py': '7feb33cb0ebe2fa58ad87f6b2131ccbe766c8bf477c3c958d10b42af92668ef2', 'council/__init__.py': '329298564fa761413630b872e45b49d744051c50f47cc06d31edf3fcb3e8c773', 'council/critic.py': 'bcc350429209f03ef199d1f5d694d16c36b9165504dbc317bbbc1cadfc5468f9', 'council/engine.py': '655b0c407c72dc7939176b85c445f5ca9034eb4dbdbfb1c80077827ce3d2359b', 'council/generator.py': '1ecd5957fa02f3db2b7992e132f354da9fe1d2bc0e652bafda10c448cc8ab4c8', 'council/runner.py': '2cbbdfadaa55740bc1d32835d21f4fd097433127ac28033c93fc69c573a2d9d4', 'council_multitarget.py': '5e738d7156ca2b43d0c810bfb659eb11b285d28724e9ff55a670ca961f8cd80f', 'council_runner.py': 'ae2565edc07abc511f2c606f54ecdc3dc52b98c5254f4ccb0c43d150a780ecab', 'gpu_context_probe.py': 'f905795ce8a768030cc9a826670cc73944be0089a19ee81e48b586d70daa8311', 'kaggle_run.py': 'db922dd85c6358f72a26e027831809f87d544cd0af88e073bee0fec3531e4de1', 'notebooks/kaggle_multitarget_smoke_v4.ipynb': '788e0f93051164c82c339218b064550ae3cc845a751fb3a6a4e146601157dafe', 'paired_study.py': '181f0d21e998daabb13041a13d13baf6bb5eb123c91bf61bd54f9454f560a84f', 'pilot/__init__.py': '09b67cf5fee1772b0d9f5dacbbe62d7ab6b11d6e4cc28f8f4266f6b2e0e52464', 'pilot/api_contract.py': '775fa5025f40852da9edd28551b377b4e28fd585e458fb9819f7ba291f606519', 'pilot/api_probe_analysis.py': '354c2777a4cf8ff12a870bdd446eba3ece37dcfda01bce7af6fa01c5dce37c19', 'pilot/attention.py': '2cbb0deb89364bbbb6908b7e88c59cdbe9f429bef4bc9eab2f4463bc8f92e855', 'pilot/backend.py': '4a367b4b5588c9fe688fe1486e0c334e55e9e112768ff2d147cf49d1336c9878', 'pilot/benchmark.py': 'dd0a38d738bd15793fcbc3d446f21ed2be27a0bfa163575f499cc7ed6700f374', 'pilot/common.py': 'ec56ff130444000176935bd9da14a50b397009aff15badce38b7c954049b7e89', 'pilot/console.py': '1f27112e62f7446ff50d8aed611e38d8bd99ba7442bd23d841cd9c4d324078fa', 'pilot/gpu_lease.py': '12505e17b0d451d1e8b89f680c8a77ce9423657ca4aa065094b0471f7bd19860', 'pilot/grounding.py': 'fe9be1949741591bcaffaacd337d75a4254ed1046f064b17f408c6463b624fa8', 'pilot/initialization_cache.py': '1e5caf4d3e4237308073a62b2ef8cfb3276798e7e20d0d48db96ed47fd3d64ec', 'pilot/instrumentation.py': '577bfde8f16fdfc2d12ec72ec3ebff7b6bcb163c1941e41f727053be8a83ef7c', 'pilot/measurement.py': '81f2d5e827f3eda5eed0cb64691645ed728bcd1fcf780d0caecfa98f725e3f54', 'pilot/model_profiles.py': '05a81e1e91f99f8fafa92d4c2e2f98f25a3e0e251c302ab46bf091d697d11083', 'pilot/module_mutation.py': 'ea8b830955629c4c8abaab43ff318ea55278e7170b866af48fe0707b531239e0', 'pilot/multitarget.py': '9f061ddaf7397a647a29a47cef30331e5aa1e8f23f6d307856619bf9085500bc', 'pilot/notebook.py': '5d94be4daa0eb2056ec9aa94d272c72915529795b215078c32d70cd3d51c42ff', 'pilot/output_contract.py': '091ac1891eaaf58fab5d129c64abf2715e3120ef2f18740cfcaa68607acb77a1', 'pilot/prescreen.py': '716a2e8f9b3d4ef913b5dbe5c48670673ffb184064ff094eb907cbb0a2ae19c2', 'pilot/probe.py': '367015a088a092aaec2021cc5f49796eae9cc6d15e7de16ed49ce4732d811c00', 'pilot/progress.py': 'b25db42a756435a67d665716a8ef6a29a4a1c008b7e06257e36833b504376675', 'pilot/quality.py': '92d31fb233a8102313368f251e83c120bd887ae81e423347bd9d39b303f82df7', 'pilot/reuse.py': 'e80134a5988fcd318a7adc8020e4fadbaa5afced356e8228094568bb4274947e', 'pilot/runtime_limits.py': 'a614bff522935b7f515440027d2d66aab2ce804da1d68fca52a5f2926d9db630', 'pilot/scheduling.py': '3b781e64f28872a5ea792d9c12eda47caa0b271a331c75191d58fc1a9a618861', 'pilot/scoring.py': '66b847795ce11523fbda9cd49c4347ac5d29be37fcddc3e18e07ed7c56a1dd5d', 'pilot/screen_trace.py': '77feb353b6f2952b91336c978481014d7e55bd1bccc633df57ce25cac6e6bcd2', 'pilot/setup.py': '9b1f1e579a6a0526f3e92813faa0e3b2a8d6c925a5f59aca22c56d69b52fb5a9', 'pilot/source_prompt.py': 'aa19865348f30a37f557fd7c1cbf2f1dfe548269baa886460dda1e3b8c4cb93f', 'pilot/telemetry.py': 'de151bd6880595c98a3bfd2cbbc1e0b26d7a13079589df70340fe1857886150f', 'pilot/upstream_adapter.py': 'dfef842a9fe5e0b8db5054da2ee77b2e45273470ae2462379fb8f7c8367dcb17', 'pilot/validate.py': 'd1e57a63496b5985c182a56025c083d3778d299f3d3cf5ba136a8710b9b890f3', 'pilot.json': 'af87638ffa25424041c9b172e11aaa160e6efce34ffcb50dad7953d86187a210', 'pytest.ini': '0d6e34350f9e9a731b52183ee69d00334a8c560d849ccc9278b1ebfe48c73a9d', 'requirements-analysis.txt': '6fe163ca5631bab1ca772c8ea624297c2b3b5b86b40d9059e7ceebf6eb717510', 'requirements-modern.txt': '97c174f60ea4a808b2ed8cfb477bcc9188df1aedc65802012cbdbfce82657fa4', 'requirements-notebook.txt': '9847fe4dc03a9a4f1ca1752873d1ed97035c75ad21f699b68fcca811d366f879', 'requirements-paired.txt': 'e603a5074d7ccd0531e5c37ebb731a96c3d038fff9c8691af81e8a01de885cf6', 'research/benchmark/api_contract_development_probe.json': '79a9e0d028fd9c6f1098381cfa0595e3e5a5d68c4f9a594201e349334b04f201', 'research/benchmark/api_contract_heldout_frame.json': 'c2ceaae4e0c8374a2e18ebc6ddbbd900f7e5f6bb0d237d12bd4aba10b1c3a9c3', 'research/benchmark/api_source_development_probe_v2.json': 'e769c5fe9037dfff7bf1f707bbeb004439f5b84f74f4537438e7ccee0a7ceeb0', 'research/benchmark/api_source_heldout_component_v2.json': 'e89de4086cf3ba2ab2d62fafe59709a792a5e129dd297c801002f404e78bf6e5', 'research/benchmark/na-projects.json': '97e0a0a272046b858fa1e46b82f8b2b0afb3d81da3f0571d84edc3d0c58d7787', 'research/benchmark/projects-upstream.json': '54bfd0a005ae780525a6c8f5051f95a69671a40e989f0b2ddfba42c5ffc4d8c5', 'resume_paired.py': '7a5516dcb6c5afefb77dbf9a7b0bde133d9a6420cba50667d7672f062e914c0a', 'run.py': '340c51d7a90296e8f842ca7a2da7d8961499e5e620f5ab963667acf5e42dc928', 'run_research.py': 'aeea0208490def7517fb3f51d5354fdadd6cd4d9a97dcf5bd7c9ff643e65ae9e', 'scripts/api_contract_probe.py': '58fa45956fd3e97bd299630c357d84474029fc3b4f9aa6f37c031ea45c6a74ae', 'scripts/check_upstream_fixture.py': '432c928989fd4d0de1c3c176f5fe8e4c7fa9caeb902c19a32b21aa8887c1a11f', 'scripts/check_upstream_initialization.py': 'd72516f78e0c43814d5c742d0a6b7a414b7fbf94c0a52498209341484dbe1e29', 'scripts/kaggle_fresh_compact.py': '82c46b6db47ed68fe70378c387f8b067158a5eda447fac73cac6ed2246c35a10', 'scripts/kaggle_quick_oom_check.py': '92bde4593ceb0357a1a7b579008e7ff606e93892ba2f43faabb52d0c337fcab3', 'setup_context.py': '44e697a8e6963ba58e19ad080ad4dd80289420c28d4f73eaf45e32548c3f9613', 'smoke_pilot.py': '39ae9ee5e07bddbc5ddce481335a1c1a56f67d9b210215e65f9bdf7c266d3853', 'tests/test_api_contract.py': '0d8829e76fbe42bcd3f847065b0fd0f7ae58a30f5189e801cd88e5cd4842d3dc', 'tests/test_api_contract_probe.py': 'a632af58237ac7afb3f7dd41daa4a62a7da8f68db577ce1cc20dd37c6294a3db', 'tests/test_api_probe_analysis.py': 'deb5ea222aa8b3534d88ffc2237acf874bb98f779afe4b7cc7ddad606e3c9a5f', 'tests/test_attention.py': 'a9769aab4c9c8029b2ad0483add6397923ef17676c5f39dce299bffcccb020d1', 'tests/test_benchmark.py': 'fe51c6b1702d16e2dd2b24d13408fc752a5cd3f65ce3ac84b7211b5ffa9397a0', 'tests/test_council.py': 'a5a2a881fc6a02c85dfaada0919db6aadbe2c7d800738ad4c7a96629d9b23696', 'tests/test_gpu_lease.py': 'acd030148885a0a288197d52314ce4789eac48051a463196fe9d581eec85d71d', 'tests/test_measurement.py': 'e1f014ed54edb8dbfa4ecdb080725ebd01ef50bac6017b42cf473bc428aa2104', 'tests/test_multitarget.py': '3df432a208590398d39f3eb3b7f2305d35c13b375169ed65448ce79b2e22341c', 'tests/test_notebook_runtime.py': '7485c0b0fb55d16ba904ba4342f1c99fc33e771221950c024c36cd12d7c86f0b', 'tests/test_output_contract.py': '211d67e97663c526b78123da93bf3df87f15bba4c2e3e9a66a7ee02c3b048d22', 'tests/test_paired.py': 'cb652fc7a004fb66110337d8372c9ac7f86f26c18b7649aaa9569deb26e460c1', 'tests/test_pilot.py': '2705fcd9bef011a5b3e0490483b43d600e1fc1b590bdac38c000de516bc59186', 'tests/test_prescreen.py': '0af81cea71e202fba5127a87243f48236a479641277f45f47588b3bb7d151833', 'tests/test_progress_counts.py': '2e5b64a726e08fe172c4adb2c2b1d23de261a581eb7cbacb477553ab3cab97ab', 'tests/test_quality.py': 'b42b9de6dd07835b2e92ad6c04d24fca7aa89e48f93cb6311c97ff1d9ac2a665', 'tests/test_research.py': '9cf673d2d2511c655efbdcffa14a7b39a44a47a08c6b5fd97c61a8fea82e522c', 'tests/test_reuse.py': '982692dfb1fe207b6550c0192f8c36d07e31bd9b66a69f436e6708834ea42b6b', 'tests/test_source_prompt.py': '8f66969b4bf0ae8bdabaadd0bf8767f1516ab48fae962b230a4b800b65f494e7', 'worker.py': 'bb9a14655f9dbecddde4fa352e05895d31e61a8f9e1306b4c145665000676aae'}
PAYLOAD='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    assert set(archive.namelist())==set(HASHES)
    for name,expected in HASHES.items():
        data=archive.read(name)
        assert hashlib.sha256(data).hexdigest()==expected,name
        path=(WORKDIR/name).resolve()
        assert path.is_relative_to(WORKDIR.resolve())
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(data)
os.chdir(WORKDIR)
sys.path.insert(0,str(WORKDIR))
PROVENANCE=dict(base_commit=BASE_COMMIT,overlay_sha256=HASHES,source_kind='Git base plus embedded local changes; not claimed pushed')
print('Source:',WORKDIR,'Study:',ROOT)


In [ ]:
# Immutable initialization evidence and an explicitly hashed launcher.
archive=Path('/kaggle/working/PRIOR_14B_INITIALIZATION_EVIDENCE.zip')
data=base64.b64decode('...')
assert hashlib.sha256(data).hexdigest()=='603266e8f6360ae0fc86d93d2bed7edc131db23a81448017a80202518d566b3a'
if archive.exists():
    assert archive.read_bytes()==data, 'Different previous evidence exists; preserve it.'
else:
    archive.write_bytes(data)
driver=WORKDIR/'scripts/kaggle_fresh_compact.py'
driver.write_text('"""Fresh-session compact small screen; no larger inference stage."""\nimport hashlib\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport zipfile\n\nparser=argparse.ArgumentParser(description=__doc__)\nparser.add_argument(\'source\',type=Path)\nparser.add_argument(\'root\',type=Path)\nparser.add_argument(\'--mechanism\',choices=[\'api-contract-v1\'])\nparser.add_argument(\'--action\',choices=[\'small\',\'probe\'],default=\'small\')\narguments=parser.parse_args()\nif arguments.action==\'probe\' and not arguments.mechanism:\n    parser.error(\'The mechanism diagnostic requires --mechanism api-contract-v1\')\nsource,root=arguments.source,arguments.root\nstatus_path = Path(\'/kaggle/working/TEST4PY_FRESH_STATUS.json\')\nstate = dict(status=\'running\', source=str(source), root=str(root), started_epoch=time.time(),\n             scope=\'API mechanism diagnostic only; no larger campaign\' if arguments.action==\'probe\'\n                   else \'Two-module compact engineering screen only; no larger campaign\')\nlog = Path(\'/kaggle/working\') / (\'FRESH_COMPACT_\'+root.name+\'.log\')\nenv = dict(os.environ, PYTHONUNBUFFERED=\'1\', USE_TF=\'0\', USE_FLAX=\'0\', TOKENIZERS_PARALLELISM=\'false\')\npython = \'/tmp/test4py-modern/bin/python\'\n\ndef save(phase):\n    state[\'phase\'] = phase\n    state[\'updated_epoch\'] = time.time()\n    status_path.write_text(json.dumps(state, indent=2))\n    if root.exists():\n        (root/\'remote-execution.json\').write_text(json.dumps(state, indent=2))\n        if not (root/\'remote-small.log\').exists():\n            (root/\'remote-small.log\').symlink_to(log)\n\ndef run(phase, args, timeout=1800):\n    save(phase)\n    print(\'STEP\',phase,flush=True)\n    result=subprocess.run(list(map(str,args)),cwd=source,env=env,check=False,timeout=timeout)\n    if result.returncode:\n        raise RuntimeError(\'Step \'+phase+\' exited \'+str(result.returncode))\n\ntry:\n    tools = Path(\'/tmp\')/(\'test4py-tools-\'+root.name)\n    run(\'bootstrap-tools\', [sys.executable,\'-m\',\'venv\',\'--without-pip\',tools], timeout=120)\n    run(\'install-uv\', [sys.executable,\'-m\',\'pip\',\'--python\',tools/\'bin/python\',\'install\',\'uv==0.6.17\'], timeout=300)\n    env[\'PATH\']=str(tools/\'bin\')+os.pathsep+env[\'PATH\']\n    run(\'bootstrap-runtime\',[sys.executable,source/\'bootstrap_modern.py\'])\n    run(\'paired-dependencies\',[python,\'-m\',\'pip\',\'install\',\'-r\',source/\'requirements-paired.txt\'])\n    policy=[\'--mechanism\',arguments.mechanism] if arguments.mechanism else [\'--test-output-contract\',\'single-test-v1\']\n    run(\'freeze-plan\',[python,source/\'paired_study.py\',\'--action\',\'plan\',\'--output\',root,\n                       \'--max-input-tokens\',\'4096\',\'--max-new-tokens\',\'512\',*policy],timeout=90)\n    shutil.copy2(source/\'notebook-source.json\',root/\'notebook-source.json\')\n    archive=Path(\'/kaggle/working/PRIOR_14B_INITIALIZATION_EVIDENCE.zip\')\n    original_sha=\'603266e8f6360ae0fc86d93d2bed7edc131db23a81448017a80202518d566b3a\'\n    if hashlib.sha256(archive.read_bytes()).hexdigest()!=original_sha:\n        raise ValueError(\'Prior evidence archive hash mismatch\')\n    previous=root/\'imported-prior-evidence\'\n    previous.mkdir(exist_ok=False)\n    with zipfile.ZipFile(archive) as z:\n        for info in z.infolist():\n            target=(previous/info.filename).resolve()\n            if not target.is_relative_to(previous.resolve()):\n                raise ValueError(\'Unsafe prior archive member\')\n            if info.is_dir():\n                target.mkdir(parents=True,exist_ok=True)\n            else:\n                target.parent.mkdir(parents=True,exist_ok=True)\n                target.write_bytes(z.read(info))\n    # Pin the same immutable real model/embedding before new preparation.\n    shutil.copy2(previous/\'resolved_models.json\',root/\'resolved_models.json\')\n    run(\'prepare-fresh-projects\',[python,source/\'paired_study.py\',\'--action\',\'prepare\',\'--output\',root])\n    importer = \'\'\'import json, subprocess, sys\nfrom pathlib import Path\nfrom paired_study import build_plan,validate_plan\nfrom pilot.reuse import import_initialization_prefix\nfrom pilot.common import write_json\nroot,previous=map(Path,sys.argv[1:])\nsaved=json.loads((root/\'plan.json\').read_text())\nplan=build_plan(saved[\'settings\'][\'max_input_tokens\'],saved[\'settings\'][\'max_new_tokens\'],saved[\'settings\'].get(\'test_output_contract\'),saved[\'settings\'].get(\'mechanism\'))\nvalidate_plan(root,plan)\nprepared=json.loads((root/\'preflight.json\').read_text())\nruntime=json.loads(subprocess.check_output([prepared[plan[\'small\'][\'project\']][\'python\'],\'-c\',\n \'import json,torch,transformers; print(json.dumps(dict(torch=torch.__version__,transformers=transformers.__version__,gpu=torch.cuda.get_device_name(0),device_count=torch.cuda.device_count())))\'],text=True).splitlines()[-1])\nimport_initialization_prefix(root,previous,plan,runtime)\nwrite_json(root/\'initialization-import-provenance.json\',dict(origin=\'Prior real greedy initialization from an earlier Kaggle session\',original_archive_sha256=\'603266e8f6360ae0fc86d93d2bed7edc131db23a81448017a80202518d566b3a\',fresh_preparation=True,calibration=\'Must run fresh two-GPU stress next; earlier-session calibration is not reused\',runtime=runtime,exclusions=\'No test generation, repair, pytest, coverage or mutant outcomes imported\'))\nprint(\'Verified cross-session initialization outputs only; fresh hardware calibration still required.\',flush=True)\n\'\'\'\n    run(\'verify-initialization-only\',[python,\'-c\',importer,root,previous],timeout=180)\n    run(\'fresh-gpu-calibration\',[python,source/\'paired_study.py\',\'--action\',\'stress\',\'--output\',root,\'--jobs\',\'2\'],timeout=3700)\n    if arguments.action==\'probe\':\n        run(\'real-api-contract-probe\',[python,source/\'scripts/api_contract_probe.py\',\'--study\',root,\n            \'--manifest\',source/\'research/benchmark/api_contract_development_probe.json\',\'--output\',root/\'diagnostic\'],timeout=3700)\n    else:\n        run(\'real-compact-small\',[python,source/\'paired_study.py\',\'--action\',\'small\',\'--output\',root,\'--jobs\',\'2\'],timeout=3700)\n    state.update(status=\'completed\',returncode=0)\nexcept BaseException as exc:\n    state.update(status=\'failed\',error=repr(exc),traceback=traceback.format_exc())\n    print(\'DRIVER ERROR\',repr(exc),flush=True)\nfinally:\n    state[\'ended_epoch\']=time.time()\n    state[\'seconds\']=state[\'ended_epoch\']-state[\'started_epoch\']\n    save(\'finished\')\n    if root.exists():\n        sys.path.insert(0,str(source))\n        from paired_study import make_bundle\n        make_bundle(root)\n    print(\'FINAL\',json.dumps(state),flush=True)\n\nraise SystemExit(0 if state["status"] == "completed" else 2)\n')
PROVENANCE.update(fresh_driver_sha256=hashlib.sha256(driver.read_bytes()).hexdigest(),
                  initialization_archive_sha256='603266e8f6360ae0fc86d93d2bed7edc131db23a81448017a80202518d566b3a')
(WORKDIR/'notebook-source.json').write_text(json.dumps(PROVENANCE,indent=2))
BOOTLOG=Path('/kaggle/working')/('FRESH_COMPACT_'+ROOT.name+'.log')
print('Fresh study:',ROOT)


In [ ]:
# Blocking, bounded job; no detached Run-All output loss.
with BOOTLOG.open('x') as log:
    proc=subprocess.Popen([sys.executable,str(driver),str(WORKDIR),str(ROOT)],
        cwd=WORKDIR,env=dict(os.environ,PYTHONUNBUFFERED='1'),stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        log.write(line);log.flush()
        if line.startswith(('STEP','FINAL','[progress:','[stress','Project ','Reports:',
                            'Verified','DRIVER ERROR')):
            print(line,end='',flush=True)
    code=proc.wait()
status=json.loads(Path('/kaggle/working/TEST4PY_FRESH_STATUS.json').read_text())
print('Execution status:',status['status'])
from IPython.display import display,FileLink,Markdown
if (ROOT/'small-report.md').exists():display(Markdown((ROOT/'small-report.md').read_text()))
if Path(str(ROOT)+'.zip').exists():display(FileLink(str(ROOT)+'.zip'))
assert code==0 and status['status']=='completed', 'Failed step; inspect preserved log/archive.'
print('Review tests, assertion edits, coverage/fault evidence and costs. Larger pilot not launched.')
